In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
client = OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com/v1")

In [3]:
# SYSTEM_PROMPT = """
# You must format every response using ONLY these tags:

# - <*&TEXT&*> for text
# - <*&CODE:filename=...:lang=...:downloadable=true|false&*> for code
# - <*&COMMAND&*> for commands
# - <*&IMAGE:path=...&*> for images
# - <*&PDF:path=...&*> for PDF files
# - <*&AUDIO:path=...&*> for audio
# - <*&YOUTUBE:url=...&*> for YouTube embeds
# - <*&END&*> to end the response (required)

# Rules:
# - ALWAYS end with <*&END&*>
# - DO NOT output anything outside tags
# - Tags are case-sensitive and must match exactly
# - You can use multiple TEXT, CODE, etc. blocks in one response
# - Maintain correct order (TEXT → CODE → TEXT → COMMAND → ...)

# CODE rules:
# - downloadable=true ONLY if the code is a complete runnable file
# - otherwise downloadable=false

# Formatting rules:
# - No spaces inside tag brackets
# - All attributes must be included exactly as shown
# - Do not invent new tags

# Examples:

# <*&TEXT&*>Here is your code:
# <*&CODE:filename=main.py:lang=python:downloadable=true&*>print("Hello")
# <*&TEXT&*>Run it:
# <*&COMMAND&*>python3 main.py
# <*&END&*>

# <*&TEXT&*>Watch this:
# <*&YOUTUBE:url=https://www.youtube.com/embed/CG48pSyK8GU&*>
# <*&END&*>
# """

In [16]:
for r in client.chat.completions.create(
    model="deepseek-reasoner",
    messages=[
        # {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "i want you to call the 2 functions you have to test tool calls"}
    ],
    stream=True,
    tools=[
      {
  "type": "function",
  "function": {
    "name": "time",
    "description": "Returns the current time as a string in formate \"%Y-%m-%d %H:%M:%S\". Used when current date and time are required. If you output to user, use a letteral format to be readable like: \"It's three oclock PM on fifth of September\" instead of numbers to be pronounced well.",
    "parameters": {
      "properties": {},
      "type": "object"
    }
  }
},
{
  "type": "function",
  "function": {
    "name": "createNewRunner",
    "description": "Use when and if you need to create a python code runner process. The runner stays open to recieve code and execute it untill you end the process. Only the runner is in sandbox but the rest of agent is out.",
    "parameters": {
      "properties": {
        "details": {
          "description": "A description you want to attach to the runner to remember what the purpose of this runner is and its details (ex. 'Create a machine learning model to detect houses pricing based on input data. the code will read '/home/user/datasets/houses.csv' file then do some preprocessing and cleaning then scaling then train KNN model then save it in '.pkl' file.').",
          "type": "string"
        }
      },
      "required": [
        "details"
      ],
      "type": "object"
    }
  }
}
]
):
    # print(r.choices[0].delta.content, end="")
    print(r)

ChatCompletionChunk(id='cad5ee2c-c087-4edc-9bea-8b6eae0e1e19', choices=[Choice(delta=ChoiceDelta(content=None, function_call=None, refusal=None, role='assistant', tool_calls=None, reasoning_content=''), finish_reason=None, index=0, logprobs=None)], created=1785702394, model='deepseek-v4-flash', object='chat.completion.chunk', service_tier=None, system_fingerprint='fp_a18b46594c_prod0820_fp8_kvcache_20260402', usage=None)
ChatCompletionChunk(id='cad5ee2c-c087-4edc-9bea-8b6eae0e1e19', choices=[Choice(delta=ChoiceDelta(content=None, function_call=None, refusal=None, role=None, tool_calls=None, reasoning_content='The'), finish_reason=None, index=0, logprobs=None)], created=1785702394, model='deepseek-v4-flash', object='chat.completion.chunk', service_tier=None, system_fingerprint='fp_a18b46594c_prod0820_fp8_kvcache_20260402', usage=None)
ChatCompletionChunk(id='cad5ee2c-c087-4edc-9bea-8b6eae0e1e19', choices=[Choice(delta=ChoiceDelta(content=None, function_call=None, refusal=None, role=None

In [12]:
from tools.PythonRunner import PythonRunner


In [6]:
# print(PythonRunner.createNewRunner.args_schema.model_json_schema())

In [13]:
from langchain_core.utils.function_calling import convert_to_openai_tool
import json

In [8]:
# convert_to_openai_tool(PythonRunner.createNewRunner)

In [9]:
# print(convert_to_openai_tool(PythonRunner.createNewRunner))

In [10]:
# from tools import time_utils

In [14]:
print(json.dumps(convert_to_openai_tool(PythonRunner.createNewRunner), indent=2))

{
  "type": "function",
  "function": {
    "name": "createNewRunner",
    "description": "Use when and if you need to create a python code runner process. The runner stays open to recieve code and execute it untill you end the process. Only the runner is in sandbox but the rest of agent is out.",
    "parameters": {
      "properties": {
        "details": {
          "description": "A description you want to attach to the runner to remember what the purpose of this runner is and its details (ex. 'Create a machine learning model to detect houses pricing based on input data. the code will read '/home/user/datasets/houses.csv' file then do some preprocessing and cleaning then scaling then train KNN model then save it in '.pkl' file.').",
          "type": "string"
        }
      },
      "required": [
        "details"
      ],
      "type": "object"
    }
  }
}


In [2]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com/v1")
history = [
    {
        'role': 'system',
        'content': 'You are a helpful assistant.'
    },
    {
        'role': 'developer',
        'content': 'Always answer briefly.'
    },
    {
        'role': 'user',
        'content': 'What is TCP?'
    },
    {
        'role': 'assistant',
        'content': 'TCP is a transport-layer protocol.'
    },
    {
        'role': 'user',
        'content': 'What about UDP?'
    }
]

response = client.responses.create(
    model='deepseek-reasoner',
    tools= [
      {
  "type": "function",
  "function": {
    "name": "time",
    "description": "Returns the current time as a string in formate \"%Y-%m-%d %H:%M:%S\". Used when current date and time are required. If you output to user, use a letteral format to be readable like: \"It's three oclock PM on fifth of September\" instead of numbers to be pronounced well.",
    "parameters": {
      "properties": {},
      "type": "object"
    }
  }
},
{
  "type": "function",
  "function": {
    "name": "createNewRunner",
    "description": "Use when and if you need to create a python code runner process. The runner stays open to recieve code and execute it untill you end the process. Only the runner is in sandbox but the rest of agent is out.",
    "parameters": {
      "properties": {
        "details": {
          "description": "A description you want to attach to the runner to remember what the purpose of this runner is and its details (ex. 'Create a machine learning model to detect houses pricing based on input data. the code will read '/home/user/datasets/houses.csv' file then do some preprocessing and cleaning then scaling then train KNN model then save it in '.pkl' file.').",
          "type": "string"
        }
      },
      "required": [
        "details"
      ],
      "type": "object"
    }
  }
}
],
    input=history
)

print('INPUT:')
print(history)

print()
print('RAW RESPONSE:')
print(response)

print()
print('OUTPUT ITEMS:')
for item in response.output:
    print(item)

BadRequestError: Error code: 400 - {'error': {'message': 'Failed to deserialize the JSON body into the target type: tools[0]: missing field `name` at line 1 column 698', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}